In [1]:
import random
import string
import csv
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter

N_IMAGES = 50_000
W, H = 168, 32

OUT_DIR = Path("dataset")
IMG_DIR = OUT_DIR / "images"
LABELS = OUT_DIR / "labels.csv"

ALPHABET = string.ascii_uppercase
MIN_LEN, MAX_LEN = 4, 8

random.seed(123)

In [2]:
FONTS = ["fonts/DejaVuSans.ttf"]

def load_font(size: int):
    return ImageFont.truetype(FONTS[0], size)

def random_word():
    n = random.randint(MIN_LEN, MAX_LEN)
    return "".join(random.choice(ALPHABET) for _ in range(n))

def render_image(text: str):
    img = Image.new("L", (W, H), 255)
    draw = ImageDraw.Draw(img)

    font_size = H - 8
    font = load_font(font_size)

    bbox = draw.textbbox((0, 0), text, font=font)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]

    while tw > W - 2 and font_size > 8:
        font_size -= 1
        font = load_font(font_size)
        bbox = draw.textbbox((0, 0), text, font=font)
        tw = bbox[2] - bbox[0]
        th = bbox[3] - bbox[1]

    x = max(1, (W - tw) // 2 - 4)
    y = max(0, (H - th) // 2 - 4)

    LETTER_SPACING = 2

    x_cursor = x
    for ch in text:
        if random.random() < 0.3:
            for dx in (0, 1):
                for dy in (0, 1):
                    draw.text((x_cursor + dx, y + dy), ch, fill=0, font=font)
        else:
            draw.text((x_cursor, y), ch, fill=0, font=font)

        ch_bbox = draw.textbbox((0, 0), ch, font=font)
        ch_w = ch_bbox[2] - ch_bbox[0]
        x_cursor += ch_w + LETTER_SPACING


    if random.random() < 0.05:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.2, 0.7)))

    if random.random() < 0.05:
        px = img.load()
        for _ in range(random.randint(10, 40)):
            px[random.randint(0, W - 1), random.randint(0, H - 1)] = \
                0 if random.random() < 0.5 else 255

    return img

In [3]:
OUT_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

with open(LABELS, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "text"])

    for i in range(N_IMAGES):
        if i % 5000 == 0:
            print(f"Generated {i} images")
        text = random_word()
        img = render_image(text)

        fname = f"{i:06d}.png"
        img.save(IMG_DIR / fname, optimize=True)
        writer.writerow([fname, text])

print("DONE")
print("Images:", IMG_DIR)
print("Labels:", LABELS)


Generated 0 images
Generated 5000 images
Generated 10000 images
Generated 15000 images
Generated 20000 images
Generated 25000 images
Generated 30000 images
Generated 35000 images
Generated 40000 images
Generated 45000 images
DONE
Images: dataset\images
Labels: dataset\labels.csv
